# Sanity check - raw signals
Salgan al sol - Billy bond y la pesada

In [2]:
%load_ext autoreload
%autoreload 2
import tools_EEG as TEEG

## Pedro's Visualization method

In [3]:
from scipy.io import loadmat
import numpy as np
file_path = "/home/tperezsanchez/FoundationModel_EEG_Dissertation/EEG_data_vis/data/Working/XB47Y/XB47Y_181.mat"
fn = file_path

m = loadmat(fn, squeeze_me=True, struct_as_record=False)

# mostrar keys útiles (ignoramos las internas de MATLAB que empiezan con "__")
keys = [k for k in m.keys() if not k.startswith("__")]
print("KEYS:", keys)

# opcional: ver tipo/shape de cada key
for k in keys:
    v = m[k]
    shp = getattr(v, "shape", None)
    print(f"{k:20s} type={type(v)} shape={shp}")

KEYS: ['hdr', 'data']
hdr                  type=<class 'scipy.io.matlab._mio5_params.mat_struct'> shape=None
data                 type=<class 'numpy.ndarray'> shape=(2, 4472028)


## Extract Fs, labels, nChans, nSamples del hdr

In [4]:
import matplotlib.pyplot as plt
m = loadmat(fn, squeeze_me=True, struct_as_record=False)

hdr = m["hdr"]
data = m["data"]

# --- leer campos del header ---
fs = float(np.array(hdr.Fs).squeeze())
nChans_hdr = int(np.array(hdr.nChans).squeeze())
nSamples_hdr = int(np.array(hdr.nSamples).squeeze())

# labels pueden venir en varios formatos; esto lo fuerza a lista de strings
labels_raw = np.array(hdr.label).squeeze()
labels = [str(x) for x in labels_raw.reshape(-1)]

print("fs:", fs)
print("nChans (hdr):", nChans_hdr)
print("nSamples (hdr):", nSamples_hdr)

print("data shape:", data.shape, "=> (nChans, nSamples)")
print("labels (len):", len(labels))
print("labels:", labels)

fs: 207.0310546581987
nChans (hdr): 2
nSamples (hdr): 4472028
data shape: (2, 4472028) => (nChans, nSamples)
labels (len): 2
labels: ['EEG SQ_D-SQ_C', 'EEG SQ_P-SQ_C']


In [6]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"   # alternativa: "notebook"

In [5]:
m = loadmat(fn, squeeze_me=True, struct_as_record=False)

hdr = m["hdr"]
X = np.asarray(m["data"], dtype=float)  # (nChans, nSamples)
fs = float(np.array(hdr.Fs).squeeze())
labels = [str(x) for x in np.array(hdr.label).squeeze().reshape(-1)]

N = X.shape[1]
t_sec = np.arange(N) / fs

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_sec, y=X[0], mode="lines", name=labels[0]))
fig.add_trace(go.Scatter(x=t_sec, y=X[1], mode="lines", name=labels[1]))

fig.update_layout(
    xaxis_title="Time (s)",
    yaxis_title="Amplitude (uV)",
    width=1100,
    height=500
)

out_html = "eeg_plot.html"
fig.write_html(out_html, include_plotlyjs="cdn")
print(f"Guardado: {out_html} (abrilo en tu navegador)")

Guardado: eeg_plot.html (abrilo en tu navegador)


In [7]:
file_path = "/home/tperezsanchez/FoundationModel_EEG_Dissertation/EEG_data_vis/data/Working/XB47Y/XB47Y_seizures.xlsx"
df_sq, df_di = TEEG.preprocess_seizure_data_1_3(file_path)
path = "//home/tperezsanchez/FoundationModel_EEG_Dissertation/EEG_data_vis/data/Working/XB47Y/"
df_XB47Y, error_list = TEEG.process_eeg_mat_files_1_1(path)
df_matches = TEEG.plot_eeg_availability_with_onsetsV2_1_4(
    df_files=df_XB47Y, 
    df_onsets=df_sq, 
    output_path="/home/tperezsanchez/FoundationModel_EEG_Dissertation/EEG_data_vis/results/XB47Y_MAPWITHSEIZ.png",
    show_plot=False
)

Processing Patient: XB47Y
Output directory: /home/tperezsanchez/FoundationModel_EEG_Dissertation/EEG_data_vis/data/Working/XB47Y/preprocessCSV_XB47Y
Successfully saved: 
 - sqEEG.csv 
 - diary.csv
--- Processing Summary ---
Successfully processed: 183
Errors encountered: 0
Total significant gaps (>1s): 78


In [9]:
import os
import numpy as np
import pandas as pd
from scipy.io import loadmat
import plotly.graph_objects as go

def plot_one_onset_html(mat_dir: str, row: pd.Series, out_dir: str = "."):
    # --- tiempos del DF ---
    startTs = pd.to_datetime(row["T0"])
    stopTs  = pd.to_datetime(row["TF"])
    onTs    = pd.to_datetime(row["onset"])

    # --- cargar .mat ---
    fn = os.path.join(mat_dir, row["file"])
    m = loadmat(fn, squeeze_me=True, struct_as_record=False)

    hdr = m["hdr"]
    X = np.asarray(m["data"], dtype=float)  # (2, N)

    fs = float(np.array(hdr.Fs).squeeze())
    labels = [str(x) for x in np.array(hdr.label).squeeze().reshape(-1)]

    N = X.shape[1]
    t_sec = np.arange(N) / fs

    # --- sample del onset ---
    ixSz = int((onTs - startTs).total_seconds() * fs)

    # clamp por seguridad
    ixSz = max(0, min(ixSz, N - 1))

    # --- plot ---
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t_sec, y=X[0], mode="lines", name=labels[0]))
    fig.add_trace(go.Scatter(x=t_sec, y=X[1], mode="lines", name=labels[1]))
    fig.add_trace(go.Scatter(
        x=[t_sec[ixSz], t_sec[ixSz]],
        y=[np.min(X), np.max(X)],
        mode="lines",
        line=dict(color="red", width=3),
        name="Seizure onset"
    ))

    # ticks como tu EDF: start / onset / stop
    fig.update_layout(
        xaxis=dict(
            tickmode="array",
            tickvals=[0, t_sec[ixSz], t_sec[-1]],
            ticktext=[str(startTs), str(onTs), str(stopTs)],
            tickangle=45,
            title="Time (s)"
        ),
        yaxis=dict(title="Amplitude (uV)"),
        width=1100,
        height=500
    )

    os.makedirs(out_dir, exist_ok=True)
    out_html = os.path.join(out_dir, f"{os.path.splitext(row['file'])[0]}__{onTs.strftime('%Y%m%d_%H%M%S')}.html")
    fig.write_html(out_html, include_plotlyjs="cdn")
    return out_html, ixSz, fs, fn

# ---- EJEMPLO DE USO ----
# df_matches = tu df
# mat_dir = "<path/a/tu/carpeta_mat>"
row = df_matches.iloc[0]
mat_dir = path

out_html, ixSz, fs, fn = plot_one_onset_html(mat_dir, row, out_dir="plots_html")
print("Saved:", out_html, "| ixSz:", ixSz, "| fs:", fs, "| file:", fn)

Saved: plots_html/XB47Y_41__20191031_232508.html | ixSz: 2721040 | fs: 207.0310546581987 | file: //home/tperezsanchez/FoundationModel_EEG_Dissertation/EEG_data_vis/data/Working/XB47Y/XB47Y_41.mat


In [6]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=[0,1,2], y=[0,1,0], mode="lines+markers"))
fig.show()